# WTA Match Predictor — EDA y Feature Engineering
Pipeline completo: carga, limpieza, clasificación de torneos, cálculo de features derivadas y generación del dataset de entrenamiento.

## 1. Imports y carga de datos

In [6]:
import pandas as pd
import numpy as np
import os
from ydata_profiling import ProfileReport

In [ ]:
#!pip install kagglehub

     ---------------------------------------- 0.0/40.1 kB ? eta -:--:--
     ------------------- ------------------ 20.5/40.1 kB 217.9 kB/s eta 0:00:01
     -------------------------------------- 40.1/40.1 kB 317.6 kB/s eta 0:00:00
   ---------------------------------------- 0.0/70.6 kB ? eta -:--:--
   ---------------------------------------- 70.6/70.6 kB 1.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/217.8 kB ? eta -:--:--
   ---------------------------------------  215.0/217.8 kB 4.4 MB/s eta 0:00:01
   ---------------------------------------- 217.8/217.8 kB 3.3 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Descarga de Kaggle actualizado
import kagglehub

# Download latest version
path = kagglehub.dataset_download("dissfya/wta-tennis-2007-2023-daily-update")

print("Path to dataset files:", path)

100%|██████████| 1.03M/1.03M [00:00<00:00, 2.34MB/s]

Extracting files...
Path to dataset files: C:\Users\NaiaJon\.cache\kagglehub\datasets\dissfya\wta-tennis-2007-2023-daily-update\versions\1062


In [8]:
# Cuidado con las rutas al copiarlo al gihub
file_path = os.path.join(path, "wta.csv")
wta = pd.read_csv(file_path, low_memory=False)
print(f'Partidos cargados: {len(wta)}')
wta.tail()

Partidos cargados: 44446


,Tournament,Date,Court,Surface,Round,Best of,Player_1,Player_2,Winner,Rank_1,Rank_2,Pts_1,Pts_2,Odd_1,Odd_2,Score
44441,Mutua Madrid Open,2026-04-29 00:00:00,Outdoor,Clay,Quarterfinals,3,Pliskova Ka.,Potapova A.,Potapova A.,197,56,375,1065,2.5,1.53,1-6 7-6 3-6
44442,Mutua Madrid Open,2026-04-29 00:00:00,Outdoor,Clay,Quarterfinals,3,Kostyuk M.,Noskova L.,Kostyuk M.,23,13,1722,2849,1.5,2.63,7-6 6-0
44443,Mutua Madrid Open,2026-04-30 00:00:00,Outdoor,Clay,Semifinals,3,Baptiste H.,Andreeva M.,Andreeva M.,32,8,1392,3746,2.63,1.5,4-6 6-7
44444,Mutua Madrid Open,2026-04-30 00:00:00,Outdoor,Clay,Semifinals,3,Kostyuk M.,Potapova A.,Kostyuk M.,23,56,1722,1065,1.33,3.4,6-2 1-6 6-1
44445,Mutua Madrid Open,2026-05-02 00:00:00,Outdoor,Clay,The Final,3,Andreeva M.,Kostyuk M.,Kostyuk M.,8,23,3746,1722,1.67,2.2,3-6 5-7


## 2. Exploración inicial

In [9]:
wta.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44446 entries, 0 to 44445
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Tournament  44446 non-null  object
 1   Date        44446 non-null  object
 2   Court       44446 non-null  object
 3   Surface     44446 non-null  object
 4   Round       44446 non-null  object
 5   Best of     44446 non-null  int64 
 6   Player_1    44446 non-null  object
 7   Player_2    44446 non-null  object
 8   Winner      44446 non-null  object
 9   Rank_1      44446 non-null  int64 
 10  Rank_2      44446 non-null  int64 
 11  Pts_1       44446 non-null  int64 
 12  Pts_2       44446 non-null  int64 
 13  Odd_1       44446 non-null  object
 14  Odd_2       44446 non-null  object
 15  Score       44446 non-null  object
dtypes: int64(5), object(11)
memory usage: 5.4+ MB


In [ ]:
# Profiling completo — genera wta_report.html
profile = ProfileReport(wta, title='WTA Profiling Report')
profile.to_file('wta_report.html')

## 3. Limpieza y transformaciones

In [10]:
# Eliminar columnas no útiles para el modelo
# Court: muy desbalanceado (40k outdoor vs 400 indoor), la superficie ya lo captura
# Best of: en WTA siempre es al mejor de 3 sets
wta = wta.drop(columns=['Court', 'Best of'])

In [11]:
# Unificar superficies: Greenset y Carpet tienen muy pocas entradas y son pistas rápidas → Hard
wta['Surface'] = wta['Surface'].replace({'Greenset': 'Hard', 'Carpet': 'Hard'})
print(wta['Surface'].value_counts())

Surface
Hard     27290
Clay     12347
Grass     4809
Name: count, dtype: int64


In [12]:
# Clasificar torneos por categoría usando palabras clave
# Los nombres cambian con patrocinadores pero la ciudad/torneo es estable

def clasificar_torneo(nombre):
    n = nombre.lower()
    
    if any(x in n for x in ['australian open', 'roland garros', 'french open',
                              'wimbledon', 'us open']):
        return 'GS'
    
    if any(x in n for x in ['wta finals', 'wta elite trophy', 'championships',
                              'tournament of champions', 'riyadh finals', 'wta tour championships']):
        return 'WTA_Finals'
    
    if any(x in n for x in ['indian wells', 'miami', 'madrid', 'roma', 'rome',
                              'internazionali', 'canada', 'toronto', 'rogers',
                              'canadian', 'cincinnati', 'beijing', 'china open',
                              'wuhan', 'doha', 'qatar', 'dubai', 'bnp paribas open',
                              'national bank open', 'sony ericsson open',
                              'western & southern financial group', 'shenzhen']):
        return 'WTA1000'
    
    if any(x in n for x in ['abu dhabi', 'abu dabi', 'adelaide', 'berlin', 'bett1open',
                              'eastbourne', 'rothesay', 'bad homburg', 'san jose',
                              'silicon valley', 'guangzhou', 'osaka', 'tokyo',
                              'toray pan pacific', 'seoul', 'strasbourg', 'birmingham',
                              'nottingham', 'brisbane', 'linz', 'merida', 'monterrey',
                              'charleston', 'stuttgart', 'porsche', 'washington',
                              'mubadala', 'guadalajara', 'gdl open akron',
                              'family circle cup', 'aegon', 'kremlin', 'pilot pen',
                              'new haven', 'bank of the west', 'sydney international',
                              'stanford', 'luxembourg', 'belgium', 'diamond games',
                              'fortis', 'san diego', 'zhengzhou', 'german']):
        return 'WTA500'
    
    return 'WTA250_o_menor'

wta['tournament_type'] = wta['Tournament'].map(clasificar_torneo)
print(wta['tournament_type'].value_counts())

tournament_type
WTA250_o_menor    14363
WTA1000           10354
GS                 9455
WTA500             8343
WTA_Finals         1931
Name: count, dtype: int64


In [13]:
# Comprobación: torneos sin clasificar
sin_clasificar = wta[wta['tournament_type'] == 'WTA250_o_menor']['Tournament'].nunique()
print(f'Torneos únicos sin clasificar: {sin_clasificar}')

Torneos únicos sin clasificar: 145


In [14]:
# Eliminar Tournament (ya tenemos tournament_type) y convertir tipos
wta = wta.drop(columns='Tournament')

wta['Date']  = pd.to_datetime(wta['Date'], errors='coerce')
wta['Odd_1'] = pd.to_numeric(wta['Odd_1'], errors='coerce')
wta['Odd_2'] = pd.to_numeric(wta['Odd_2'], errors='coerce')

print('Nulos tras conversión:')
print(wta[['Date', 'Odd_1', 'Odd_2']].isna().sum())

Nulos tras conversión:
Date     2
Odd_1    1
Odd_2    1
dtype: int64


In [15]:
# Eliminar partidos sin fecha 
wta = wta.dropna(subset=['Date'])

In [16]:
# Limpiar odds: valores < 1 son códigos de sin datos (equivalente a -1)
# Los convertimos a NaN — se usarán para comparar con casas pero no para entrenar
wta.loc[wta['Odd_1'] < 1, 'Odd_1'] = None
wta.loc[wta['Odd_2'] < 1, 'Odd_2'] = None

In [17]:
# Calcular probabilidades implícitas de las casas de apuestas
# Indican lo que se paga por euro la victoria (2.5, 3.5... etc). Lo paso a probabilidades
# 1/odd y normalizar para que sumen 1 (elimina el margen de la casa: suman un poco más de uno)

wta['prob_1'] = 1 / wta['Odd_1']
wta['prob_2'] = 1 / wta['Odd_2']
total = wta['prob_1'] + wta['prob_2']
wta['prob_1'] = wta['prob_1'] / total
wta['prob_2'] = wta['prob_2'] / total

print(wta['prob_1'].describe())

count    44328.000000
mean         0.498446
std          0.214081
min          0.019231
25%          0.337349
50%          0.500000
75%          0.662651
max          0.980769
Name: prob_1, dtype: float64


In [18]:
# Convertir tipos de columnas categóricas y de texto
wta['Surface']  = wta['Surface'].astype(str)
wta['Round']    = wta['Round'].astype(str)
wta['Score']    = wta['Score'].astype(str)
wta['Player_1'] = wta['Player_1'].astype(str)
wta['Player_2'] = wta['Player_2'].astype(str)
wta['Winner']   = wta['Winner'].astype(str)

In [19]:
# Crear target: 1 si gana Player_1, 2 si gana Player_2
def asignar_target(fila):
    if fila['Winner'] == fila['Player_1']:
        return 1
    else:
        return 0

wta['target'] = wta.apply(asignar_target, axis=1)
print(wta['target'].value_counts())

target
1    22222
0    22222
Name: count, dtype: int64


In [20]:
# Guardar wta limpio para uso en app.py y consultas
wta.to_csv('wta_limpio.csv', index=False)
print('wta_limpio.csv guardado')

wta_limpio.csv guardado


In [26]:
wta['Round'].unique()

array(['1st Round', '2nd Round', 'Quarterfinals', 'Semifinals',
       'The Final', '3rd Round', '4th Round', 'Round Robin',
       'Third Place'], dtype=object)

## 4. Feature Engineering

Calculamos features derivadas para cada partido usando solo información disponible **antes** de ese partido (corte temporal para evitar data leakage).

In [21]:
def forma_reciente(df, jugadora, fecha_limite, meses=2):
    """Win rate de una jugadora en los N meses anteriores a la fecha.
    Devuelve 0 si no tiene partidos (lesión o pausa larga)"""
    fecha_inicio = fecha_limite - pd.DateOffset(months=meses)
    mask = (
        ((df['Player_1'] == jugadora) | (df['Player_2'] == jugadora)) &
        (df['Date'] < fecha_limite) &
        (df['Date'] >= fecha_inicio)
    )
    partidos = df[mask]
    if len(partidos) == 0:
        return 0
    victorias = (partidos['Winner'] == jugadora).sum()
    return victorias/len(partidos)

In [22]:
def winrate(df, jugadora, fecha_limite, superficie=None, ronda=None):
    """Win rate histórico de una jugadora, filtrable por superficie y/o ronda.
    Devuelve 0.4 si no tiene historial (ligeramente por debajo de neutro = novata en esa condición)"""
    mask = (
        ((df['Player_1'] == jugadora) | (df['Player_2'] == jugadora)) &
        (df['Date'] < fecha_limite)
    )
    if superficie:
        mask &= (df['Surface'] == superficie)
    if ronda:
        mask &= (df['Round'] == ronda)
    partidos = df[mask]
    if len(partidos) == 0:
        return 0.4
    victorias = (partidos['Winner'] == jugadora).sum()
    return victorias/len(partidos)

In [23]:
def headtohead(df, p1, p2, fecha_limite):
    """% de victorias de p1 sobre p2 en enfrentamientos directos previos a la fecha.
    Devuelve 0.5 si nunca se han enfrentado (neutro)"""
    mask = (
        (
            ((df['Player_1'] == p1) & (df['Player_2'] == p2)) |
            ((df['Player_1'] == p2) & (df['Player_2'] == p1))
        ) & (df['Date'] < fecha_limite)
    )
    partidos = df[mask]
    if len(partidos) == 0:
        return 0.5
    victorias_p1 = (partidos['Winner'] == p1).sum()
    return victorias_p1/len(partidos)

In [24]:
def experiencia(df, jugadora, fecha_limite):
    """Número total de partidos jugados por la jugadora antes de la fecha"""
    mask = (
        ((df['Player_1'] == jugadora) | (df['Player_2'] == jugadora)) &
        (df['Date'] < fecha_limite)
    )
    return df[mask].shape[0]

In [25]:
# Comprobación de las funciones antes de ejecutar el bucle completo
jugadora = 'Sabalenka A.'
fecha_limite = pd.to_datetime('2025-12-17')
print(f"Experiencia: {experiencia(wta, jugadora, fecha_limite)} partidos")
print(f"H2H vs Badosa: {headtohead(wta, jugadora, 'Badosa P.', fecha_limite):.2%}")
print(f"Win rate Clay: {winrate(wta, jugadora, fecha_limite, superficie='Clay'):.2%}")
print(f"Forma reciente: {forma_reciente(wta, jugadora, fecha_limite):.2%}")

Experiencia: 491 partidos
H2H vs Badosa: 71.43%
Win rate Clay: 73.68%
Forma reciente: 80.00%


### Construcción del dataset de features

⚠️ Este bucle tarda ~30 minutos. El resultado se guarda en CSV para no tener que repetirlo.

In [ ]:
# Reset index para usar como match_id
wta = wta.reset_index()

features = []
for i, partido in wta.iterrows():
    p1    = partido['Player_1']
    p2    = partido['Player_2']
    fecha = partido['Date']
    
    row = {
        # Identificador y metadata
        'match_id': i,
        'date':     partido['Date'],
        'odd_1':    partido['prob_1'],
        'odd_2':    partido['prob_2'],

        # Features del partido
        'surface':         partido['Surface'],
        'round':           partido['Round'],
        'tournament_type': partido['tournament_type'],

        # Features calculadas (solo info anterior al partido)
        'rank_diff':           float(partido['Rank_1']) - float(partido['Rank_2']),
        'wins2meses_p1':       forma_reciente(wta, p1, fecha),
        'wins2meses_p2':       forma_reciente(wta, p2, fecha),
        'ratio_superficie_p1': winrate(wta, p1, fecha, superficie=partido['Surface']),
        'ratio_superficie_p2': winrate(wta, p2, fecha, superficie=partido['Surface']),
        'h2h':                 headtohead(wta, p1, p2, fecha),
        'ratio_ronda_p1':      winrate(wta, p1, fecha, ronda=partido['Round']),
        'ratio_ronda_p2':      winrate(wta, p2, fecha, ronda=partido['Round']),
        'experiencia_p1':      experiencia(wta, p1, fecha),
        'experiencia_p2':      experiencia(wta, p2, fecha),

        # Target
        'target': partido['target']
    }
    features.append(row)

historico_partidos = pd.DataFrame(features)

# Feature de inexperiencia
historico_partidos['is_new_p1'] = (historico_partidos['experiencia_p1'] < 10).astype(int)
historico_partidos['is_new_p2'] = (historico_partidos['experiencia_p2'] < 10).astype(int)

print(f'Dataset construido: {len(historico_partidos)} partidos')
print(historico_partidos.info())

## 5. Limpieza del dataset de features

In [ ]:
# Eliminar 2007: primer año del dataset, las features derivadas están muy incompletas
# porque no hay historial previo para calcularlas
historico_partidos = historico_partidos[historico_partidos['date'].dt.year != 2007]
print(f'Partidos tras eliminar 2007: {len(historico_partidos)}')

In [ ]:
# Verificar nulos
print(historico_partidos.isnull().sum())

In [ ]:
# Guardar — carga posterior con pd.read_csv('historico_partidos.csv', parse_dates=['date'])
historico_partidos.to_csv('historico_partidos.csv', index=False)
print('✓ historico_partidos.csv guardado')

# Clustering. Perfiles de jugadoras

In [2]:
stats_2024 = pd.read_csv(r'C:\Users\NaiaJon\Documents\Naia\BootcampDataScience\Datos ML\wta_matches_2024.csv', low_memory=False)

In [3]:
stats_2024.columns

Index(['tourney_id', 'tourney_name', 'surface', 'draw_size', 'tourney_level',
       'tourney_date', 'match_num', 'winner_id', 'winner_seed', 'winner_entry',
       'winner_name', 'winner_hand', 'winner_ht', 'winner_ioc', 'winner_age',
       'loser_id', 'loser_seed', 'loser_entry', 'loser_name', 'loser_hand',
       'loser_ht', 'loser_ioc', 'loser_age', 'score', 'best_of', 'round',
       'minutes', 'w_ace', 'w_df', 'w_svpt', 'w_1stIn', 'w_1stWon', 'w_2ndWon',
       'w_SvGms', 'w_bpSaved', 'w_bpFaced', 'l_ace', 'l_df', 'l_svpt',
       'l_1stIn', 'l_1stWon', 'l_2ndWon', 'l_SvGms', 'l_bpSaved', 'l_bpFaced',
       'winner_rank', 'winner_rank_points', 'loser_rank', 'loser_rank_points'],
      dtype='object')

In [4]:
stats_2024.head()

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
0,2024-9900,United Cup,Hard,18,I,20240101,299,216347,NaN,NaN,...,27.0,13.0,7.0,7.0,4.0,8.0,1.0,9505.0,NaN,NaN
1,2024-9900,United Cup,Hard,18,I,20240101,297,216347,NaN,NaN,...,47.0,28.0,7.0,12.0,3.0,8.0,1.0,9505.0,20.0,2330.0
2,2024-9900,United Cup,Hard,18,I,20240101,295,201493,NaN,NaN,...,82.0,47.0,10.0,15.0,11.0,19.0,NaN,NaN,292.0,246.0
3,2024-9900,United Cup,Hard,18,I,20240101,293,216347,NaN,NaN,...,27.0,18.0,6.0,8.0,6.0,11.0,1.0,9505.0,14.0,2770.0
4,2024-9900,United Cup,Hard,18,I,20240101,291,201614,NaN,NaN,...,72.0,51.0,22.0,16.0,3.0,6.0,20.0,2330.0,544.0,89.0
